# Day 2: Guided Lab — Prompt Engineering with Structured Outputs

## Learning Objectives

By the end of this lab, you will be able to:

1. **Structure prompts** using the 5-component framework
2. **Apply few-shot learning** to improve output consistency
3. **Use chain-of-thought** prompting for complex reasoning
4. **Generate guaranteed-valid JSON** using Pydantic schemas
5. **Compare prompt variations** systematically with evaluation metrics

## Prerequisites

- Completed Day 1 labs
- Basic understanding of LLM parameters (temperature, tokens)
- Google AI Studio API key

---

## Part 0: Setup and Infrastructure

In [ ]:
# Install required packages
!pip install -q -U google-genai

In [ ]:
# Import libraries
import os
import time
import json
import pandas as pd
from datetime import datetime, timezone
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Configure API — reads the GEMINI_API_KEY you set up on Day 1
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except:
    API_KEY = None

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Enter your Gemini API key: ")

# Initialize client
client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-2.5-flash-lite"

print(f"✓ API key loaded")
print(f"✓ Using model: {MODEL_ID}")

In [ ]:
# Prompt logging infrastructure
PROMPT_LOG = []

def _now():
    """Get current timestamp (timezone-aware)."""
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate(prompt, temperature=0.7, max_tokens=1000, log=True, label=None):
    """
    Generate text using Gemini (free-form output).
    
    Args:
        prompt: The prompt text
        temperature: Creativity (0=deterministic, 1=creative)
        max_tokens: Maximum response length
        log: Whether to log this call
        label: Optional label for this experiment
    
    Returns:
        The generated text
    """
    start_time = time.time()
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens,
        ),
    )
    
    result = response.text
    latency = time.time() - start_time
    
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(),
            "label": label,
            "type": "free_form",
            "prompt": prompt[:500] + "..." if len(prompt) > 500 else prompt,
            "prompt_length": len(prompt),
            "temperature": temperature,
            "response": result[:500] + "..." if len(result) > 500 else result,
            "response_length": len(result),
            "latency_s": round(latency, 3)
        })
    
    return result

def generate_structured(prompt, schema_model, temperature=0.2, log=True, label=None):
    """
    Generate structured output using Pydantic schema.
    
    The API GUARANTEES the output matches your schema - no parsing errors!
    
    Args:
        prompt: The prompt text
        schema_model: Pydantic BaseModel class defining the output structure
        temperature: Creativity (lower is better for structured output)
        log: Whether to log this call
        label: Optional label for this experiment
    
    Returns:
        Parsed Pydantic model instance
    """
    start_time = time.time()
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_json_schema": schema_model.model_json_schema(),
        },
    )
    
    latency = time.time() - start_time
    raw_text = response.text or ""
    
    # Parse into Pydantic model (guaranteed to work with structured outputs)
    result = schema_model.model_validate_json(raw_text)
    
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(),
            "label": label,
            "type": "structured",
            "schema": schema_model.__name__,
            "prompt": prompt[:500] + "..." if len(prompt) > 500 else prompt,
            "prompt_length": len(prompt),
            "temperature": temperature,
            "response": raw_text[:500] + "..." if len(raw_text) > 500 else raw_text,
            "response_length": len(raw_text),
            "latency_s": round(latency, 3)
        })
    
    return result

def show_log():
    """Display the prompt log as a DataFrame."""
    if not PROMPT_LOG:
        print("No prompts logged yet.")
        return None
    return pd.DataFrame(PROMPT_LOG)

print("✓ Logging infrastructure ready")

---

## Part 1: The Anatomy of a Prompt

A well-structured prompt can have up to **5 components**:

1. **Context** — Background information, role assignment
2. **Instructions** — What task to perform
3. **Input** — The specific data to process
4. **Examples** — Demonstrations of desired output
5. **Constraints** — Boundaries, format requirements

Let's see how adding components improves output quality.

### Exercise 1.1: Basic vs. Structured Prompts

Let's compare a basic prompt with a structured one for the same task.

In [ ]:
# The task: Summarize a customer review
review = """
I bought this wireless keyboard last month and have mixed feelings. The typing 
experience is excellent - the keys are responsive and quiet, which is perfect 
for my home office. Battery life has been impressive too, still on the original 
batteries after 4 weeks of daily use. However, the Bluetooth connection drops 
occasionally, maybe once or twice a day, which is frustrating during video calls 
when I'm trying to type in chat. The build quality feels a bit cheap for the $80 
price point. Overall, it's a decent keyboard but not without its quirks.
"""

# Version 1: Basic prompt (Instructions + Input only)
basic_prompt = f"Summarize this review:\n\n{review}"

# Version 2: Structured prompt (all 5 components)
structured_prompt = f"""You are a product analyst at an e-commerce company.

Task: Analyze the following customer review and provide a structured summary.

Review:
---
{review}
---

Provide your analysis in this format:
- Sentiment: [positive/negative/mixed]
- Pros: [bullet points]
- Cons: [bullet points]
- Key insight: [one sentence]

Keep the summary under 100 words."""

# Compare them
print("=" * 60)
print("BASIC PROMPT RESPONSE:")
print("=" * 60)
print(generate(basic_prompt, temperature=0.3, label="basic"))

print("\n" + "=" * 60)
print("STRUCTURED PROMPT RESPONSE:")
print("=" * 60)
print(generate(structured_prompt, temperature=0.3, label="structured"))

### 💡 Discussion

Notice how the structured prompt produces:
- Consistent format
- Specific categories (pros/cons)
- Actionable insights
- Appropriate length

The basic prompt works, but results vary more between runs.

### Exercise 1.2: The Power of Role Assignment

Let's see how different roles change the output for the same question.

In [ ]:
question = "What should I consider when choosing a cloud provider for my startup?"

roles = {
    "no_role": f"{question}",
    
    "cto": f"""You are an experienced CTO who has built multiple startups from scratch.
{question}""",
    
    "cfo": f"""You are a CFO focused on cost optimization and financial planning.
{question}""",
    
    "security": f"""You are a cybersecurity expert specializing in cloud security.
{question}"""
}

for role, prompt in roles.items():
    print(f"\n{'='*60}")
    print(f"ROLE: {role.upper()}")
    print("="*60)
    response = generate(prompt, temperature=0.5, max_tokens=300, label=f"role_{role}")
    print(response[:600])

---

## Part 2: Few-Shot Prompting

**Few-shot prompting** provides examples of desired input-output pairs. This is often more effective than detailed instructions.

> "Show, don't just tell"

### Exercise 2.1: Zero-Shot vs. Few-Shot Classification

In [ ]:
# Task: Classify customer support tickets
ticket = "My order #12345 arrived but one item was missing from the package."

# Zero-shot: Just instructions
zero_shot = f"""Classify this customer support ticket into one of these categories:
- billing
- shipping
- product_issue
- account
- other

Ticket: {ticket}

Category:"""

# Few-shot: Instructions + examples
few_shot = f"""Classify customer support tickets into categories.

Ticket: "I was charged twice for my subscription this month."
Category: billing

Ticket: "The laptop screen has dead pixels."
Category: product_issue

Ticket: "I can't reset my password, the email never arrives."
Category: account

Ticket: "When will my package arrive? It's been 2 weeks."
Category: shipping

Ticket: "{ticket}"
Category:"""

print("Zero-shot result:", generate(zero_shot, temperature=0, label="zero_shot_classify"))
print("Few-shot result:", generate(few_shot, temperature=0, label="few_shot_classify"))

---

## Part 3: Chain-of-Thought Prompting

For complex reasoning tasks, asking the model to "think step by step" dramatically improves accuracy.

**Why it works:** More output tokens = more computation = better reasoning

### Exercise 3.1: Direct vs. Chain-of-Thought

In [ ]:
# Math problem
problem = """
A store sells notebooks for $3 each. If you buy 10 or more, you get 15% off.
Sales tax is 8%. How much does it cost to buy 12 notebooks?
"""

# Direct approach
direct_prompt = f"{problem}\nAnswer:"

# Chain-of-thought approach
cot_prompt = f"{problem}\nLet's solve this step by step:"

print("DIRECT ANSWER:")
print(generate(direct_prompt, temperature=0, label="math_direct"))

print("\n" + "="*60)
print("\nCHAIN-OF-THOUGHT:")
print(generate(cot_prompt, temperature=0, label="math_cot"))

---

## Part 4: Structured Outputs with Pydantic 🔥

This is the **key technique** for production applications!

Instead of hoping the model returns valid JSON, we use **Pydantic schemas** with the API's `response_json_schema` parameter. The API **guarantees** the output matches our schema.

### Exercise 4.1: Define a Schema

In [ ]:
# Define the output structure using Pydantic
class PersonInfo(BaseModel):
    """Extracted information about a person."""
    name: str = Field(description="Full name of the person")
    age: Optional[int] = Field(description="Age in years, or null if not mentioned")
    occupation: Optional[str] = Field(description="Job title or profession")
    location: Optional[str] = Field(description="City or location mentioned")
    employer: Optional[str] = Field(description="Company or employer name")

# Test extraction
text = "John Smith is a 35-year-old software engineer living in San Francisco. He works at Google."

prompt = f"""Extract information about the person from this text.
If information is not mentioned, use null.

Text: {text}"""

result = generate_structured(prompt, PersonInfo, label="person_extraction")

# Result is already a Pydantic object - no parsing needed!
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")
print(f"Location: {result.location}")
print(f"Employer: {result.employer}")
print(f"\nFull object: {result.model_dump_json(indent=2)}")

### Exercise 4.2: Classification with Constrained Categories

Use `Literal` types to constrain outputs to specific values.

In [ ]:
# Define schema with constrained categories
class TicketClassification(BaseModel):
    """Classification result for a support ticket."""
    ticket_id: str = Field(description="ID of the ticket")
    category: Literal["billing", "shipping", "product_issue", "account", "other"] = Field(
        description="Primary category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="Urgency level"
    )
    summary: str = Field(description="One-sentence summary of the issue")
    suggested_action: str = Field(description="Recommended next step")

# Test tickets
tickets = [
    {"id": "T001", "text": "I was charged twice for my order last week!"},
    {"id": "T002", "text": "Package says delivered but I never received it."},
    {"id": "T003", "text": "How do I update my email address on my account?"},
]

for ticket in tickets:
    prompt = f"""Classify this support ticket.

Urgency guidelines:
- high: Financial issues, security concerns, service outages
- medium: Delayed orders, missing items, access issues
- low: General questions, feedback, minor issues

Ticket ID: {ticket['id']}
Ticket: {ticket['text']}"""
    
    result = generate_structured(prompt, TicketClassification, label=f"classify_{ticket['id']}")
    print(f"\n{ticket['id']}: {result.category} ({result.urgency})")
    print(f"  → {result.suggested_action}")

### Exercise 4.3: Batch Extraction

Process multiple items in a single API call using a batch schema.

In [ ]:
# Schema for a single review analysis
class ReviewAnalysis(BaseModel):
    """Analysis of a single product review."""
    review_id: str
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    confidence: float = Field(description="Confidence score 0-1")
    key_points: List[str] = Field(description="Key points from the review (max 3)")

# Batch schema wraps multiple analyses
class ReviewBatch(BaseModel):
    """Batch of review analyses."""
    reviews: List[ReviewAnalysis]

# Sample reviews
reviews = [
    {"id": "R1", "text": "Amazing product! Works perfectly and arrived fast."},
    {"id": "R2", "text": "Terrible quality. Broke after one week. Waste of money."},
    {"id": "R3", "text": "It's okay. Does what it says, nothing special."},
    {"id": "R4", "text": "Love the features but the battery life is disappointing."},
]

# Format reviews for prompt
reviews_text = "\n".join([f"{r['id']}: {r['text']}" for r in reviews])

prompt = f"""Analyze these product reviews.

Reviews:
{reviews_text}

For each review, determine sentiment, confidence, and extract up to 3 key points."""

batch_result = generate_structured(prompt, ReviewBatch, label="batch_reviews")

# Convert to DataFrame for nice display
df = pd.DataFrame([r.model_dump() for r in batch_result.reviews])
print(df)

### Exercise 4.4: Sanity Checks

Even with structured outputs, validate business rules!

In [ ]:
def validate_reviews(reviews: List[ReviewAnalysis]) -> List[tuple]:
    """Run sanity checks on review analyses."""
    problems = []
    
    for r in reviews:
        # Check confidence is in valid range
        if not 0 <= r.confidence <= 1:
            problems.append((r.review_id, "invalid_confidence", r.confidence))
        
        # Check key_points isn't too long
        if len(r.key_points) > 3:
            problems.append((r.review_id, "too_many_key_points", len(r.key_points)))
        
        # Check for empty key points on non-neutral reviews
        if r.sentiment != "neutral" and len(r.key_points) == 0:
            problems.append((r.review_id, "missing_key_points", r.sentiment))
    
    return problems

# Validate our batch results
problems = validate_reviews(batch_result.reviews)
print(f"Validation problems found: {len(problems)}")
for p in problems:
    print(f"  {p}")

---

## Part 5: Prompt Iteration with Evaluation

Let's build a complete workflow: extract data, evaluate against golden labels, and iterate.

In [ ]:
# Define schema for support ticket triage
class TriageResult(BaseModel):
    """Triage result for a support ticket."""
    id: str = Field(description="Ticket ID")
    category: Literal["Bug", "Policy", "Request", "Complaint", "Other"] = Field(
        description="Primary category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="Urgency level"
    )
    summary: str = Field(description="One-sentence summary (max 20 words)")
    next_step: str = Field(description="Recommended action")

class TriageBatch(BaseModel):
    """Batch of triage results."""
    tickets: List[TriageResult]

In [ ]:
# Test data
support_tickets = [
    {"id": "T1", "text": "App crashes when I try to upload photos. Tried reinstalling, still broken."},
    {"id": "T2", "text": "Can you add dark mode? Would really help with eye strain."},
    {"id": "T3", "text": "What's your refund policy for annual subscriptions?"},
    {"id": "T4", "text": "Your service has been down for 3 hours! I'm losing business!"},
    {"id": "T5", "text": "How do I export my data to CSV?"},
    {"id": "T6", "text": "I was promised a discount but was charged full price."},
]

# Golden set (human-labeled ground truth)
GOLDEN = {
    "T1": {"category": "Bug", "urgency": "high"},
    "T2": {"category": "Request", "urgency": "low"},
    "T3": {"category": "Policy", "urgency": "low"},
    "T4": {"category": "Bug", "urgency": "high"},
    "T5": {"category": "Policy", "urgency": "low"},
    "T6": {"category": "Complaint", "urgency": "medium"},
}

In [ ]:
# Version 1: Basic prompt
PROMPT_V1 = """You are a customer support assistant.

Triage these support tickets into categories and urgency levels.

Tickets:
{tickets}
"""

def format_tickets(tickets):
    return "\n".join([f"{t['id']}: {t['text']}" for t in tickets])

def run_triage(prompt_template, tickets, label):
    prompt = prompt_template.format(tickets=format_tickets(tickets))
    return generate_structured(prompt, TriageBatch, label=label)

# Run v1
result_v1 = run_triage(PROMPT_V1, support_tickets, "triage_v1")
pred_v1 = {t.id: t for t in result_v1.tickets}

# Show results
df_v1 = pd.DataFrame([t.model_dump() for t in result_v1.tickets])
print("V1 Results:")
print(df_v1[["id", "category", "urgency", "summary"]])

In [ ]:
# Evaluation function
def evaluate(predictions, golden, field):
    """Calculate accuracy for a field."""
    correct = 0
    total = 0
    errors = []
    
    for id, expected in golden.items():
        if id in predictions:
            total += 1
            pred_value = getattr(predictions[id], field)
            if pred_value == expected[field]:
                correct += 1
            else:
                errors.append((id, expected[field], pred_value))
    
    accuracy = correct / total if total > 0 else 0
    return {"correct": correct, "total": total, "accuracy": accuracy, "errors": errors}

# Evaluate v1
cat_eval = evaluate(pred_v1, GOLDEN, "category")
urg_eval = evaluate(pred_v1, GOLDEN, "urgency")

print(f"V1 Category Accuracy: {cat_eval['correct']}/{cat_eval['total']} = {cat_eval['accuracy']:.1%}")
print(f"V1 Urgency Accuracy:  {urg_eval['correct']}/{urg_eval['total']} = {urg_eval['accuracy']:.1%}")

if cat_eval['errors']:
    print(f"\nCategory errors:")
    for id, expected, got in cat_eval['errors']:
        print(f"  {id}: expected {expected}, got {got}")

In [ ]:
# Version 2: Improved prompt with explicit rules
PROMPT_V2 = """You are a customer support assistant.

Triage these support tickets into categories and urgency levels.

CATEGORY RULES:
- Bug: System errors, crashes, broken features
- Request: Feature requests, enhancement suggestions
- Policy: Questions about pricing, refunds, how things work
- Complaint: Billing disputes, service issues, frustration
- Other: Anything else

URGENCY RULES:
- high: Service outages, data loss, financial impact, angry customers
- medium: Broken features affecting workflow, billing issues
- low: Questions, suggestions, minor issues

Tickets:
{tickets}
"""

# Run v2
result_v2 = run_triage(PROMPT_V2, support_tickets, "triage_v2")
pred_v2 = {t.id: t for t in result_v2.tickets}

# Evaluate v2
cat_eval_v2 = evaluate(pred_v2, GOLDEN, "category")
urg_eval_v2 = evaluate(pred_v2, GOLDEN, "urgency")

print(f"V2 Category Accuracy: {cat_eval_v2['correct']}/{cat_eval_v2['total']} = {cat_eval_v2['accuracy']:.1%}")
print(f"V2 Urgency Accuracy:  {urg_eval_v2['correct']}/{urg_eval_v2['total']} = {urg_eval_v2['accuracy']:.1%}")

# Compare
print(f"\n📈 Category improvement: {cat_eval['accuracy']:.1%} → {cat_eval_v2['accuracy']:.1%}")
print(f"📈 Urgency improvement:  {urg_eval['accuracy']:.1%} → {urg_eval_v2['accuracy']:.1%}")

---

## Summary and Key Takeaways

### What We Learned

| Technique | When to Use | Key Benefit |
|-----------|-------------|-------------|
| **Structured prompts** | Always | Consistent, predictable outputs |
| **Role assignment** | Domain expertise needed | Appropriate tone and focus |
| **Few-shot examples** | Format control, classification | Shows rather than tells |
| **Chain-of-thought** | Complex reasoning | More accurate answers |
| **Pydantic schemas** | Production applications | Guaranteed valid JSON |
| **Golden set evaluation** | Iteration | Measurable improvement |

### The Prompt Engineering Checklist

- [ ] Is there a clear **role/context**?
- [ ] Are **instructions** specific and unambiguous?
- [ ] Would **examples** help clarify the expected output?
- [ ] For complex reasoning, did I ask for **step-by-step** thinking?
- [ ] Is the **output schema** defined with Pydantic?
- [ ] Are **constraints** (categories, lengths) enforced in the schema?
- [ ] Do I have a **golden set** for evaluation?

In [ ]:
# Export your experiment log
if PROMPT_LOG:
    df = pd.DataFrame(PROMPT_LOG)
    df.to_csv("day2_guided_lab_log.csv", index=False)
    print(f"✓ Saved {len(PROMPT_LOG)} experiments to day2_guided_lab_log.csv")
    print(df[["label", "type", "latency_s", "response_length"]].to_string())

---

## Next Steps

In the **Independent Lab**, you will:
- Build your own extraction schema for a business domain
- Create a golden set with 8+ labeled examples
- Iterate on prompts and measure improvement
- Write error analysis

**Proceed to: Day 2 Independent Lab →**